# 03 — Adnotare Manuală Crops Parcuri

Review **interactiv** al crops-urilor auto-clasificate de B2 (pseudo-labels).

**Workflow:**
1. Fiecare crop e afișat cu clasa curentă (pseudo-label)
2. Confirmi clasa corectă cu butoanele de mai jos
3. La final: re-split → rebuild mixed_cls → retrain B3

**Clase:** `glass | metal | other | paper | plastic`

> Dacă nu ești sigur, alege **Skip** — crop-ul rămâne în clasa curentă.

In [2]:
import shutil
import json
from pathlib import Path
from IPython.display import display, clear_output
import ipywidgets as widgets

REPO_ROOT  = Path('../..').resolve()
CROPS_DIR  = REPO_ROOT / 'datasets' / 'parks_cls_unsorted' / 'all'
REVIEW_LOG = REPO_ROOT / 'datasets' / 'parks_cls_unsorted' / 'review_log.json'
CLASSES    = ['glass', 'metal', 'other', 'paper', 'plastic']
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

# Colectează toate crops-urile cu clasa curentă
all_crops = []
for cls in CLASSES:
    cls_dir = CROPS_DIR / cls
    if not cls_dir.exists():
        continue
    for p in sorted(cls_dir.iterdir()):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            all_crops.append({'path': p, 'current_cls': cls, 'assigned_cls': cls})

# Încarcă log anterior (dacă există) — rezumă de unde am rămas
reviewed = set()
if REVIEW_LOG.exists():
    with REVIEW_LOG.open(encoding='utf-8') as f:
        log_data = json.load(f)
    reviewed = set(log_data.get('reviewed', []))
    print(f'Progres anterior: {len(reviewed)}/{len(all_crops)} crop-uri review-uite')
else:
    log_data = {'reviewed': [], 'moves': []}

# Crops rămase de review
pending = [c for c in all_crops if str(c['path']) not in reviewed]

print(f'Total crops: {len(all_crops)}')
print(f'De review: {len(pending)}')
for cls in CLASSES:
    n = sum(1 for c in all_crops if c['current_cls'] == cls)
    print(f'  {cls:<10} {n:>4}')

Total crops: 284
De review: 284
  glass        12
  metal        92
  other         3
  paper       136
  plastic      41


In [3]:
from IPython.display import Image as IPImage

if not pending:
    print('[OK] Toate crop-urile au fost review-uite! Rulează celula de aplicare.')
else:
    state = {'idx': 0, 'moves': list(log_data.get('moves', []))}

    # ── Widgets ───────────────────────────────────────────────────────────────
    counter_lbl  = widgets.Label()
    current_lbl  = widgets.HTML()
    img_out      = widgets.Output(layout=widgets.Layout(width='320px', height='320px'))

    btn_skip = widgets.Button(description='Skip (OK)', button_style='success',
                              layout=widgets.Layout(width='110px'))
    class_btns = [widgets.Button(description=cls,
                                 layout=widgets.Layout(width='100px'),
                                 button_style='warning' if cls != 'other' else 'info')
                  for cls in CLASSES]
    btn_back = widgets.Button(description='← Înapoi', button_style='',
                              layout=widgets.Layout(width='100px'))
    btn_finish = widgets.Button(description='Salvează & Ieși', button_style='danger',
                                layout=widgets.Layout(width='150px'))

    top_row    = widgets.HBox([counter_lbl, current_lbl])
    action_row = widgets.HBox([btn_skip] + class_btns)
    nav_row    = widgets.HBox([btn_back, btn_finish])
    ui         = widgets.VBox([top_row, img_out, action_row, nav_row])

    def save_log():
        log_data['reviewed'] = list(reviewed | {str(pending[i]['path'])
                                                 for i in range(state['idx'])})
        log_data['moves'] = state['moves']
        with REVIEW_LOG.open('w', encoding='utf-8') as f:
            json.dump(log_data, f, indent=2)

    def show_current():
        idx = state['idx']
        if idx >= len(pending):
            with img_out:
                clear_output()
                print('[DONE] Toate crop-urile review-uite!')
            counter_lbl.value = f'{len(pending)}/{len(pending)} complete'
            return
        crop = pending[idx]
        counter_lbl.value = f'{idx+1}/{len(pending)}  '
        current_lbl.value = (f'<b>Clasa curentă: '
                             f'<span style="color:#e67e22">{crop["current_cls"]}</span></b>'
                             f'  <small style="color:#888">{crop["path"].name}</small>')
        with img_out:
            clear_output(wait=True)
            display(IPImage(filename=str(crop['path']), width=300))

    def on_skip(b):
        reviewed.add(str(pending[state['idx']]['path']))
        state['idx'] += 1
        show_current()

    def make_class_handler(target_cls):
        def handler(b):
            crop = pending[state['idx']]
            src_cls = crop['current_cls']
            if target_cls != src_cls:
                state['moves'].append({
                    'file': crop['path'].name,
                    'from': src_cls,
                    'to': target_cls
                })
            reviewed.add(str(crop['path']))
            state['idx'] += 1
            show_current()
        return handler

    def on_back(b):
        if state['idx'] > 0:
            state['idx'] -= 1
            p = str(pending[state['idx']]['path'])
            reviewed.discard(p)
            state['moves'] = [m for m in state['moves']
                               if m['file'] != pending[state['idx']]['path'].name]
            show_current()

    def on_finish(b):
        save_log()
        corrections = len(state['moves'])
        print(f'[SALVAT] {len(reviewed)} review-uite, {corrections} corecții. '
              f'Rulează celula de aplicare.')

    btn_skip.on_click(on_skip)
    btn_back.on_click(on_back)
    btn_finish.on_click(on_finish)
    for btn, cls in zip(class_btns, CLASSES):
        btn.on_click(make_class_handler(cls))

    show_current()
    display(ui)

---
## Aplicare corecții
Mută fișierele corectate în noul subfolder de clasă. **Rulează după ce ai terminat review-ul.**

In [4]:
if not REVIEW_LOG.exists():
    print('[SKIP] Niciun review log găsit. Rulează celula de review mai întâi.')
else:
    with REVIEW_LOG.open(encoding='utf-8') as f:
        log = json.load(f)

    moves = log.get('moves', [])
    print(f'Corecții de aplicat: {len(moves)}')

    applied, skipped = 0, 0
    for move in moves:
        src = CROPS_DIR / move['from'] / move['file']
        dst = CROPS_DIR / move['to']   / move['file']
        if not src.exists():
            # poate deja mutat
            if dst.exists():
                skipped += 1
                continue
            print(f'  [WARN] Lipsă: {src}')
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(src), str(dst))
        print(f'  {move["from"]} → {move["to"]}  {move["file"]}')
        applied += 1

    print(f'\nAplicat: {applied}  |  Deja OK: {skipped}')
    print('\nDistribuție după corecții:')
    for cls in CLASSES:
        n = sum(1 for p in (CROPS_DIR / cls).iterdir()
                if p.is_file() and p.suffix.lower() in IMAGE_EXTS) if (CROPS_DIR / cls).exists() else 0
        print(f'  {cls:<10} {n:>4}')

[SKIP] Niciun review log găsit. Rulează celula de review mai întâi.


---
## Rebuild parks_cls → mixed_cls → Retrain B3
**Rulează după ce ai aplicat corecțiile.**

In [5]:
import sys
sys.path.insert(0, str(REPO_ROOT))

from scripts.split_classification_dataset import main as _split_cls
from scripts.merge_classification_datasets import main as _merge_cls
from scripts.report_classification_dataset_stats import main as _report_cls

PARKS_CLS = REPO_ROOT / "datasets" / "parks_cls"
TRASHNET  = REPO_ROOT / "datasets" / "trashnet_cls"
MIXED_CLS = REPO_ROOT / "datasets" / "mixed_cls"
MANIFEST  = REPO_ROOT / "datasets" / "parks_cls_unsorted" / "crops_manifest.csv"

# 1. Re-split parks_cls
print("── 1. Split parks_cls ──────────────────────────────────")
sys.argv = [
    "split_classification_dataset.py",
    "--source-root", str(CROPS_DIR),
    "--out-root",    str(PARKS_CLS),
    "--manifest",    str(MANIFEST),
    "--group-by", "source-image",
    "--val", "0.15", "--test", "0.15", "--seed", "42", "--copy", "--clear",
]
_split_cls()

# 2. Rebuild mixed_cls
print("── 2. Merge → mixed_cls ────────────────────────────────")
sys.argv = [
    "merge_classification_datasets.py",
    "--datasets", str(TRASHNET), str(PARKS_CLS),
    "--out-dir", str(MIXED_CLS),
]
_merge_cls()

# 3. Statistici
print("── 3. Statistici mixed_cls ─────────────────────────────")
sys.argv = ["report_classification_dataset_stats.py", "--data", str(MIXED_CLS)]
_report_cls()

print("[OK] Dataset rebuilt. Rulează celula de antrenare B3.")


> d:\TrashDetectionSystem\.venv\Scripts\python.exe scripts/split_classification_dataset.py --source-root D:\TrashDetectionSystem\datasets\parks_cls_unsorted\all --out-root D:\TrashDetectionSystem\datasets\parks_cls --manifest D:\TrashDetectionSystem\datasets\parks_cls_unsorted\crops_manifest.csv --group-by source-image --val 0.15 --test 0.15 --seed 42 --copy --clear
[INFO] Split: train
 - glass: 7
 - metal: 62
 - other: 1
 - paper: 101
 - plastic: 31
[INFO] Split: val
 - glass: 3
 - metal: 12
 - other: 1
 - paper: 18
 - plastic: 6
[INFO] Split: test
 - glass: 2
 - metal: 18
 - other: 1
 - paper: 17
 - plastic: 4

> d:\TrashDetectionSystem\.venv\Scripts\python.exe scripts/merge_classification_datasets.py --datasets D:\TrashDetectionSystem\datasets\trashnet_cls D:\TrashDetectionSystem\datasets\parks_cls --out-dir D:\TrashDetectionSystem\datasets\mixed_cls
Classes: ['glass', 'metal', 'other', 'paper', 'plastic']
Removing existing output: D:\TrashDetectionSystem\datasets\mixed_cls

Merging

In [6]:
# Antrenare B3 cu date curate
from ultralytics import YOLO

model = YOLO("yolov8n-cls.pt")
results = model.train(
    data=str(MIXED_CLS),
    epochs=80,
    imgsz=224,
    batch=32,
    workers=0,
    device="0",
    patience=25,
    name="parks-cls-B3-clean",
    project=str(REPO_ROOT / "runs" / "classify"),
    seed=42,
)

# Validare pe split val după antrenare
model.val(data=str(MIXED_CLS), split="val", imgsz=224, batch=32, device="0")

print("[DONE] B3 clean antrenat!")


> d:\TrashDetectionSystem\.venv\Scripts\python.exe scripts/train_classifier.py --model yolov8n-cls.pt --data D:\TrashDetectionSystem\datasets\mixed_cls --epochs 80 --imgsz 224 --batch 32 --workers 0 --device 0 --patience 25 --name parks-cls-B3-clean --val
New https://pypi.org/project/ultralytics/8.4.25 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.23  Python-3.12.13 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\TrashDetectionSystem\datasets\mixed_cls, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=

In [7]:
# Evaluare B3-clean vs B2
import json as _json
from ultralytics import YOLO
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

CLASSES   = ["glass", "metal", "other", "paper", "plastic"]
IMG_EXTS  = {".jpg", ".jpeg", ".png", ".bmp"}
EVAL_ROOT = REPO_ROOT / "runs" / "classify_eval"


def evaluate_classifier(model_path, data_root, split, name):
    split_dir = data_root / split
    items = [
        (p, cls)
        for cls in CLASSES
        for p in sorted((split_dir / cls).iterdir())
        if p.is_file() and p.suffix.lower() in IMG_EXTS
    ] if split_dir.exists() else []
    if not items:
        print(f"  [SKIP] Nicio imagine în {split_dir}"); return

    mdl = YOLO(str(model_path))
    cls_map = (
        {int(k): v for k, v in mdl.names.items()}
        if isinstance(mdl.names, dict)
        else {i: v for i, v in enumerate(mdl.names)}
    )

    y_true, y_pred = [], []
    for img_path, true_label in items:
        res  = mdl.predict(str(img_path), imgsz=224, device="0", workers=0, verbose=False)[0]
        pred = cls_map.get(int(res.probs.top1), str(res.probs.top1))
        y_true.append(true_label)
        y_pred.append(pred)

    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1c, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=CLASSES, average=None, zero_division=0)
    mp, mr, mf1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=CLASSES, average="macro", zero_division=0)

    out_dir = EVAL_ROOT / name / split
    out_dir.mkdir(parents=True, exist_ok=True)
    summary = {
        "split": split, "num_images": len(items),
        "accuracy": acc, "macro_precision": mp,
        "macro_recall": mr, "macro_f1": mf1,
        "per_class": {
            l: {"precision": float(p), "recall": float(r),
                "f1": float(f), "support": int(s)}
            for l, p, r, f, s in zip(CLASSES, prec, rec, f1c, sup)
        },
    }
    (out_dir / "summary.json").write_text(_json.dumps(summary, indent=2), encoding="utf-8")
    print(f"  Accuracy: {acc:.4f}  |  Macro F1: {mf1:.4f}")
    return summary


for label, model_rel in [
    ("B2",       "runs/classify/parks-cls-B2/weights/best.pt"),
    ("B3-clean", "runs/classify/parks-cls-B3-clean/weights/best.pt"),
]:
    print(f"\n=== {label} ===")
    evaluate_classifier(
        model_path=REPO_ROOT / model_rel,
        data_root=MIXED_CLS,
        split="test",
        name=f"eval-{label.lower()}-mixed-test-clean",
    )



=== B2 (trashnet) ===
> d:\TrashDetectionSystem\.venv\Scripts\python.exe scripts/evaluate_classifier.py --model D:\TrashDetectionSystem\runs\classify\parks-cls-B2\weights\best.pt --data D:\TrashDetectionSystem\datasets\mixed_cls --split test --imgsz 224 --device 0 --workers 0 --name eval-b2-mixed-test-clean
[INFO] Split: test
[INFO] Images: 299
[INFO] Accuracy: 0.9331
[INFO] Macro precision: 0.9062
[INFO] Macro recall: 0.9086
[INFO] Macro F1: 0.9073
[INFO] Predictions CSV: D:\TrashDetectionSystem\runs\classify_eval\eval-b2-mixed-test-clean\test\predictions.csv
[INFO] Confusion matrix CSV: D:\TrashDetectionSystem\runs\classify_eval\eval-b2-mixed-test-clean\test\confusion_matrix.csv
[INFO] Summary JSON: D:\TrashDetectionSystem\runs\classify_eval\eval-b2-mixed-test-clean\test\summary.json


=== B3-clean (mixed) ===
> d:\TrashDetectionSystem\.venv\Scripts\python.exe scripts/evaluate_classifier.py --model D:\TrashDetectionSystem\runs\classify\parks-cls-B3-clean\weights\best.pt --data D:\Tr